In [ ]:
!git pull

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import string


from sklearn.model_selection import train_test_split

In [ ]:
%cd C:/Users/arpit/spring-2026-deep-learning-fragmented-id-resolution

In [ ]:
!git checkout Pedro

In [ ]:
# Pull ncvoters.tsv from Pedro branch without overwriting
import subprocess
import os

# Extract the file from Pedro branch and save it locally
subprocess.run([
    'git', 'show', 'Pedro:data/raw/ncvoters.tsv'
], stdout=open('data/raw', 'w'), cwd='.')

print("✓ Extracted ncvoters.tsv from Pedro branch → data/raw/ncvoters_from_Pedro.tsv")

# Read the TSV files into the DataFrame
# Data Source Hasso Plattner Institut 
# NDPL - Non-duplicates
# DPL - Duplicates
# ncvoters - a snap shot of the snapshot: VR_Snapshot_20181106 
df_ncvoters = pd.read_csv(r'data/raw/ncvoters.tsv', sep='\t')
DPL = pd.read_csv(r'data/raw/ncvoters_DPL.tsv', sep='\t')
NDPL = pd.read_csv(r'data/raw/ncvoters_NDPL.tsv', sep='\t')

print(f"✓ Loaded ncvoters: {df_ncvoters.shape}")
print(f"✓ Loaded DPL: {DPL.shape}")
print(f"✓ Loaded NDPL: {NDPL.shape}")


In [ ]:
# Preprocessing for our fragemented ID analysis 
# Selecting identifyer variables not related to voting
df_ncvoters_frag_ID = df_ncvoters[[
#Stable Identifiers
#Only to be used for labeling /evaluation only 
# / not as a model feature
    'id', 'ncid', 'voter_reg_num', 
# Primary Model features - Strongest features, Strong entropy, Essential for matching
    #many variable such as name prefx and sufx are sparse we can either use none for missing or 0/1
    'first_name', 'midl_name', 'last_name', 'name_sufx_cd',
    #other varaiabes
# Adress Similarity features - Address is the second strongest identity anchor, 
    # Street name especially high discriminative signal # Unit numbers distinguish household
    #many variable such as unit designator are sparse we can either use none for missing or 0/1
    'house_num', 'street_name', 'street_dir', 'street_type_cd', 'unit_designator', 'unit_num', 'zip_code', 'res_city_desc',
#other varaiabes
# Demographic agreement indicators - Moderate/Supporting Features 
    # these will help us reduce false matches # they are agreement indicators, low-weight similarity features
    #age group rather than age because grouped/bin age is more stable
    'age', 'age_group', 'sex', 'race_code', 'race_desc', 'ethnic_code', 'ethnic_desc', 'birth_place',
# #other varaiabes 
 'phone_num','area_cd'   
 ]]
# print("\nSelected variables 'A' and 'C':")
print(df_ncvoters_frag_ID)

In [ ]:
# Adding labels to DPL and NDPL
DPL['label'] = 1 
NDPL['label'] = 0


print(DPL.head(10))
print(NDPL.head(10))

In [ ]:
#Merge Duplicate and Non Duplicate pairs
pairs = pd.concat([DPL, NDPL])

print(pairs.head(20))
print(pairs.tail(20))


In [ ]:
# merging ncvoters on DPL and NDPL
pairs = pairs.merge(
    df_ncvoters_frag_ID,
    left_on="id1",
    right_on="id",
    how="left"
)
print(pairs.info())
print(pairs.head(10))

In [ ]:
#merging id2 
pairs = pairs.merge(
    df_ncvoters_frag_ID,
    left_on="id2",
    right_on="id",
    how="left",
    suffixes=("_1", "_2")
)

print(pairs.info())
print(pairs.head(10))

Train Test Val Split

In [ ]:
import networkx as nx
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import train_test_split 

# only duplicates
dup_pairs = pairs[pairs["label"] == 1]

G = nx.Graph()

# add edges
G.add_edges_from(zip(dup_pairs["id1"], dup_pairs["id2"]))

# connected components = entity clusters
components = list(nx.connected_components(G))

In [ ]:
# Adding ids back in data 

all_ids = set(df_ncvoters_frag_ID["id"])
ids_in_graph = set(G.nodes())

singletons = all_ids - ids_in_graph

for s in singletons:
    components.append({s})

In [ ]:
# Split

train_groups, temp_groups = train_test_split(
    components,
    test_size=0.3,
    random_state=42
)

val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.5,
    random_state=42
)

In [ ]:
#Converting groups to ids 
train_ids = set().union(*train_groups)
val_ids   = set().union(*val_groups)
test_ids  = set().union(*test_groups)

In [ ]:
#bringing dups and non dups together

train_pairs = pairs[pairs["id1"].isin(train_ids) & pairs["id2"].isin(train_ids)]

val_pairs = pairs[pairs["id1"].isin(val_ids) & pairs["id2"].isin(val_ids)]

test_pairs = pairs[pairs["id1"].isin(test_ids) & pairs["id2"].isin(test_ids)]

In [ ]:
print("Train:", len(train_pairs))
print("Val:", len(val_pairs))
print("Test:", len(test_pairs))

In [ ]:
print(df_ncvoters_frag_ID.columns)
print(pairs.columns)

print(train_pairs.columns)
print(val_pairs.columns)
print(test_pairs.columns)



 Augmented positives
 +  typos
  + nicknames
  + address abbreviation

 Hard negatives
  + same household
  + same same lastname
  + same firstname
  + suffix

Medium negative 
  + same first name negative  

Positives
  + same household
  + original duplicates
  +	typo duplicates
  + nickname duplicates
  + street abbreviation duplicates
  + multi-field duplicates

Negatives
 + household pairs
 + nickname negatives
 + suffix negatives
 + address formatting negatives


In [ ]:
#Creating a new variable id_to_clusters to be used in augmented training cell - #4 household negatives 
id_to_cluster = {}
for cluster_id, component in enumerate(components):
    for voter_id in component:
        id_to_cluster[voter_id] = cluster_id

# singletons
next_id = len(components)
for v_id in df_ncvoters_frag_ID["id"]:
    if v_id not in id_to_cluster:
        id_to_cluster[v_id] = next_id
        next_id += 1

print(f"Total IDs mapped: {len(id_to_cluster)}")

In [ ]:
# 1. Utilities (names + street logic)

import random

nickname_dict = {
    "william": ["bill","billy","will"],
    "robert": ["bob","bobby","rob"],
    "james": ["jim","jimmy"],
    "john": ["jack","johnny"],
    "elizabeth": ["liz","beth","lizzy"],
    "margaret": ["maggie","meg","peggy"],
    "katherine": ["kate","kathy"],
    "sarah": ["sara"]
}

street_expand = {
    "st":"street",
    "rd":"road",
    "ave":"avenue",
    "dr":"drive",
    "ln":"lane",
    "blvd":"boulevard"
}

def nickname(name):
    n = str(name).lower()
    if n in nickname_dict:
        return random.choice(nickname_dict[n])
    for k,v in nickname_dict.items():
        if n in v:
            return k
    return name


def typo(name):
    name=list(str(name))
    if len(name)<3:
        return "".join(name)
    i=random.randint(0,len(name)-2)
    name[i],name[i+1]=name[i+1],name[i]
    return "".join(name)


def expand_street(st):
    s=str(st).lower()
    return street_expand.get(s,s)

    

In [ ]:

# 2. Duplicate augmentation (label = 1) - This creates multi-field realistic duplicates.

def augment_duplicates(train_pairs):
    augmented = []
    for _, row in train_pairs.iterrows():
        if row["label"] != 1:
            continue
        r = row.copy()
        side1_changed = False
        side2_changed = False

        if random.random() < 0.4:
            if random.random() < 0.5:
                r["first_name_1"] = typo(r["first_name_1"])
                side1_changed = True
            else:
                r["first_name_2"] = typo(r["first_name_2"])
                side2_changed = True

        if random.random() < 0.25:
            if not side1_changed and random.random() < 0.5:
                r["first_name_1"] = nickname(r["first_name_1"])
                side1_changed = True
            elif not side2_changed:
                r["first_name_2"] = nickname(r["first_name_2"])
                side2_changed = True

        if random.random() < 0.25:
            if not side1_changed:
                r["street_type_cd_1"] = expand_street(r["street_type_cd_1"])
            elif not side2_changed:
                r["street_type_cd_2"] = expand_street(r["street_type_cd_2"])

        augmented.append(r)
    return pd.DataFrame(augmented)

#aug_dups2 = augment_duplicates(train_pairs)
aug_dups2 = augment_duplicates(train_pairs)
print(aug_dups2.shape)
print(aug_dups2["first_name_1"].head())
print(train_pairs[train_pairs["label"]==1]["first_name_1"].head())

In [ ]:
#3 adding steert_ type expansion in both direction 

def street_type_negatives(train_pairs, n=2000):
    pairs = []
    for _, row in train_pairs.sample(len(train_pairs)).iterrows():
        if row["label"] == 1:
            continue
        # skip if both street types are empty
        if row["street_type_cd_1"] == "" and row["street_type_cd_2"] == "":
            continue
        r = row.copy()
        if random.random() < 0.5:
            r["street_type_cd_1"] = expand_street(r["street_type_cd_1"])
        else:
            r["street_type_cd_2"] = expand_street(r["street_type_cd_2"])
        if r["street_type_cd_1"] != row["street_type_cd_1"] or \
           r["street_type_cd_2"] != row["street_type_cd_2"]:
            r["label"] = 0
            pairs.append(r)
        if len(pairs) >= n:
            break
    return pd.DataFrame(pairs)

In [ ]:
# ####We wanted: different people, same address --> confusing negative
#Reality:   same address → almost always same person in this dataset


# household_negatives isn't a useful hard negative generator for this specific dataset.
#  # 4. Hard negatives - We generate confusing but non-duplicate pairs. 
# # Household negatives (same address)
# # since this is being generated, I want to make sure to not accidentally append a known duplicate - cluster saftey
# def household_negatives(train_pairs,cluster_map, n=3000):

#     pairs=[]

#     groups=train_pairs.groupby(
#         ["house_num_1","street_name_1"]
#     )

#     for _,g in groups:

#         if len(g)<2:
#             continue

#         sample=g.sample(min(len(g),2))

#         r=sample.iloc[0].copy()

#         #cluster saftey check 
#         if cluster_map.get(sample.iloc[0]["id1"]) == cluster_map.get(sample.iloc[1]["id1"]):
#             continue

#         r["first_name_2"]=sample.iloc[1]["first_name_1"]
#         r["last_name_2"]=sample.iloc[1]["last_name_1"]

#         r["label"]=0
        

#         pairs.append(r)

#         if len(pairs)>=n:
#             break

#     return pd.DataFrame(pairs)

In [ ]:
#5 Nickname negatives 
#this simply adds nickname negatives by swapping some first names to nick names in the NDPL

def nickname_negatives(train_pairs,n=2000):

    pairs=[]

    for _,row in train_pairs.sample(len(train_pairs)).iterrows():
        if row["label"]==1: 
            continue

        r=row.copy()

        r["first_name_2"]=nickname(r["first_name_2"])

        if r["first_name_2"]!=row["first_name_2"]:

            r["label"]=0
            pairs.append(r)

        if len(pairs)>=n:
            break

    return pd.DataFrame(pairs)

In [ ]:
# # Suffix negatives Jr/Sr) - 0 in the negative data so this is meaningless it just would add noise and not signal 
# There 124 dups with suffixes — they're valid duplicates where one record has a missing suffix 
# #generating nonsensical suffix 
# def suffix_negatives(train_pairs,n=1000):

#     pairs=[]

#     for _,row in train_pairs.iterrows():
#          if row["label"] == 1:  # skip duplicates, work from negatives
#             continue
#         r=row.copy()

#         if pd.notna(row["name_sufx_cd_1"]):

#             r["name_sufx_cd_2"]="jr" if row["name_sufx_cd_1"]!="jr" else "sr"
#             r["label"]=0

#             pairs.append(r)

#         if len(pairs)>=n:
#             break

#     return pd.DataFrame(pairs)



Tested Suffix and it looks like it doesn't make sense to generate negative suffixes. 
* Because only dups would have junior-senior pairs. 
* could have non dups that are junior singleton and senior singleton. 
* the occurences of suffixes are sparse in our data
* we have 124 duplicate pairs with suffix mismatch -> valid raw data 

In [ ]:
# # Checking for confusable suffix cases in negatives
# confusable_negs = train_pairs[
#     (train_pairs["label"] == 0) &
#     (train_pairs["first_name_1"] == train_pairs["first_name_2"]) &
#     (train_pairs["last_name_1"] == train_pairs["last_name_2"]) &
#     (train_pairs["name_sufx_cd_1"] != train_pairs["name_sufx_cd_2"])
# ].shape[0]

# # Check confusable suffix cases in duplicates
# confusable_dups = train_pairs[
#     (train_pairs["label"] == 1) &
#     (train_pairs["first_name_1"] == train_pairs["first_name_2"]) &
#     (train_pairs["last_name_1"] == train_pairs["last_name_2"]) &
#     (train_pairs["name_sufx_cd_1"] != train_pairs["name_sufx_cd_2"])
# ].shape[0]

# print(f"Confusable suffix negatives: {confusable_negs}")
# print(f"Confusable suffix duplicates: {confusable_dups}")

# # Check how common suffixes are overall
# print(f"\nSuffix value counts id1:\n{train_pairs['name_sufx_cd_1'].value_counts()}")
# print(f"\nSuffix value counts id2:\n{train_pairs['name_sufx_cd_2'].value_counts()}")

In [ ]:
# #how many are missing suffixes on one side?
# # Verify - how many are just missing suffix on one side
# missing_one_side = suspicious[
#     (suspicious["name_sufx_cd_1"] == "") | 
#     (suspicious["name_sufx_cd_2"] == "")
# ].shape[0]

# actual_conflict = suspicious[
#     (suspicious["name_sufx_cd_1"] != "") & 
#     (suspicious["name_sufx_cd_2"] != "") &
#     (suspicious["name_sufx_cd_1"] != suspicious["name_sufx_cd_2"])
# ].shape[0]

# print(f"Missing suffix on one side: {missing_one_side}")
# print(f"Actual SR/JR conflicts: {actual_conflict}")

In [ ]:
# augmented training set 
aug_dups = augment_duplicates(train_pairs)

# hard 
street_neg = street_type_negatives(train_pairs)
#household_neg = household_negatives(train_pairs, id_to_cluster)
nickname_neg = nickname_negatives(train_pairs)

train_pairs_aug = pd.concat(
    [train_pairs,
     aug_dups, 
     street_neg,
    #household_neg, 
     nickname_neg],
    ignore_index=True
)

In [ ]:
#adding augmentation to val and test splits 
def augment_split(pairs):
    aug = augment_duplicates(pairs)
    street = street_type_negatives(pairs)
    nick = nickname_negatives(pairs)
    
    return pd.concat(
        [pairs, aug, street, nick],
        ignore_index=True
    )

# Apply consistently across all splits
# train_pairs_aug = augment_split(train_pairs)
val_pairs_aug   = augment_split(val_pairs)
test_pairs_aug  = augment_split(test_pairs)

Checking that all augmentations and hard negatives changes worked on TRAINING SET 

In [ ]:
# 1. Typo check - first_name should differ from original
typo_changes = aug_dups2[aug_dups2["first_name_1"] != train_pairs.loc[aug_dups2.index, "first_name_1"].values]
print(f"Typo applied to first_name_1: {len(typo_changes)}")

typo_changes2 = aug_dups2[aug_dups2["first_name_2"] != train_pairs.loc[aug_dups2.index, "first_name_2"].values]
print(f"Typo applied to first_name_2: {len(typo_changes2)}")

# 2. Nickname check - first_name should be in nickname dict values
nickname_changes1 = aug_dups2[aug_dups2["first_name_1"].str.lower().isin(
    [n for names in nickname_dict.values() for n in names] + list(nickname_dict.keys())
)]
print(f"Nickname in first_name_1: {len(nickname_changes1)}")

nickname_changes2 = aug_dups2[aug_dups2["first_name_2"].str.lower().isin(
    [n for names in nickname_dict.values() for n in names] + list(nickname_dict.keys())
)]
print(f"Nickname in first_name_2: {len(nickname_changes2)}")

# 3. Street expansion check - should see full words not abbreviations
expanded1 = aug_dups2[aug_dups2["street_type_cd_1"].isin(street_expand.values())]
print(f"Street expanded in street_type_cd_1: {len(expanded1)}")

expanded2 = aug_dups2[aug_dups2["street_type_cd_2"].isin(street_expand.values())]
print(f"Street expanded in street_type_cd_2: {len(expanded2)}")

# 4. Nickname negatives check
nick_negs = train_pairs_aug[
    (train_pairs_aug["label"] == 0) &
    (train_pairs_aug["first_name_2"].str.lower().isin(
        [n for names in nickname_dict.values() for n in names]
    ))
]
print(f"Nickname negatives: {len(nick_negs)}")

# 5. Street type negatives check
street_negs = train_pairs_aug[
    (train_pairs_aug["label"] == 0) &
    (train_pairs_aug["street_type_cd_1"].isin(street_expand.values()) |
     train_pairs_aug["street_type_cd_2"].isin(street_expand.values()))
]
print(f"Street type negatives: {len(street_negs)}")

# # 6. Household negatives check
# household_negs = train_pairs_aug[
#     (train_pairs_aug["label"] == 0) &
#     (train_pairs_aug["house_num_1"] == train_pairs_aug["house_num_2"]) &
#     (train_pairs_aug["street_name_1"] == train_pairs_aug["street_name_2"])
# ]
# print(f"Household negatives: {len(household_negs)}")

# 7. Overall label distribution
counts = train_pairs_aug["label"].value_counts()
pcts = train_pairs_aug["label"].value_counts(normalize=True) * 100
print(pd.DataFrame({"count": counts, "percentage": pcts.round(2)}))

In [ ]:
#How hard are our negatives?

# Check nickname negatives - how often do names match after nickname swap
nick_neg_rows = train_pairs_aug[
    (train_pairs_aug["label"] == 0) &
    (train_pairs_aug["first_name_2"].str.lower().isin(
        [n for names in nickname_dict.values() for n in names]
    ))
]

# How many have same last name - harder if last name also matches
same_last = (nick_neg_rows["last_name_1"] == nick_neg_rows["last_name_2"]).sum()
diff_last = (nick_neg_rows["last_name_1"] != nick_neg_rows["last_name_2"]).sum()

print(f"Nickname negatives with same last name: {same_last} ({same_last/len(nick_neg_rows)*100:.1f}%)")
print(f"Nickname negatives with different last name: {diff_last} ({diff_last/len(nick_neg_rows)*100:.1f}%)")

# Street negatives hardness
street_neg_rows = train_pairs_aug[
    (train_pairs_aug["label"] == 0) &
    (train_pairs_aug["street_type_cd_1"].isin(street_expand.values()) |
     train_pairs_aug["street_type_cd_2"].isin(street_expand.values()))
]

same_name = (
    (street_neg_rows["first_name_1"] == street_neg_rows["first_name_2"]) &
    (street_neg_rows["last_name_1"] == street_neg_rows["last_name_2"])
).sum()

print(f"\nStreet negatives with same full name: {same_name} ({same_name/len(street_neg_rows)*100:.1f}%)")

In [ ]:
# do truly hard negatives even EXIST in our data? No

# Same last name + nickname first name pairs from different clusters
truly_hard = train_pairs[
    (train_pairs["label"] == 0) &
    (train_pairs["last_name_1"] == train_pairs["last_name_2"]) &
    (train_pairs["first_name_1"].str.lower().isin(nickname_dict.keys())) &
    (train_pairs["first_name_2"].str.lower().isin(
        [n for names in nickname_dict.values() for n in names]
    ))
]
print(f"Truly hard nickname negatives available: {len(truly_hard)}")

What actually differ in real duplicates 

* first_name: 139 differ (2.0%) <- almost never differs
* last_name: 965 differ (14.1%) <- Secondary signal 
* house_num: 6230 differ (91.3%)  <- Primary fragmentation signal 
* street_name: 6203 differ (90.9%) <- Primary fragmentation signal 
* street_type_cd: 5082 differ (74.5%) <- Stron signal

This tells us what we knew already from the EDA 
* duplicates in our dataset are primarily people who moved — same person, different address.
* and name changes (marraige, divorce) is a secondary signal


Our baseline features should heavily weight name similarity and treat address as a weak or noisy signal rather than a strong one. The fragmented ID problem in this dataset is fundamentally an address change detection problem, not a name variation problem. 



In [ ]:
# What actually differs between real duplicates
orig_dups = train_pairs[train_pairs["label"] == 1]
cols = ["first_name", "last_name", "house_num", "street_name", "street_type_cd"]

for col in cols:
    diff = (orig_dups[f"{col}_1"] != orig_dups[f"{col}_2"]).sum()
    pct = diff / len(orig_dups) * 100
    print(f"{col}: {diff} differ ({pct:.1f}%)")

In [ ]:
# # Check how many household pairs get rejected by cluster safety check
# groups = train_pairs.groupby(["house_num_1","street_name_1"])
# rejected = 0
# eligible = 0

# for _, g in groups:
#     if len(g) < 2:
#         continue
#     sample = g.sample(min(len(g), 2))
#     eligible += 1
#     if id_to_cluster.get(sample.iloc[0]["id1"]) == id_to_cluster.get(sample.iloc[1]["id1"]):
#         rejected += 1

# print(f"Eligible groups: {eligible}")
# print(f"Rejected by cluster safety check: {rejected}")
# print(f"Passed: {eligible - rejected}")

In [ ]:
# viewing augmentation results to make 
cols=[
"first_name_1","first_name_2",
"last_name_1","last_name_2",
"house_num_1","house_num_2",
"street_name_1","street_name_2",
"street_type_cd_1","street_type_cd_2",
"label" #, "age"
]

train_pairs_aug[cols].sample(40)

In [ ]:
#Checking for nonesensical dups
train_pairs[train_pairs.label==1][
["first_name_1","first_name_2","last_name_1","last_name_2"]
].sample(40)

In [ ]:
# Check all columns are present in augmented data
print(aug_dups.columns.tolist())

# Spot check - view all fields for a few augmented records
aug_dups.sample(5).T  # .T transposes to see all fields vertically

In [ ]:
#applying across splits 
def augment_split(pairs):
    aug = augment_duplicates(pairs)
    street = street_type_negatives(pairs)
    nick = nickname_negatives(pairs)
    
    return pd.concat(
        [pairs, aug, street, nick],
        ignore_index=True
    )

# Apply consistently across all splits
#train_pairs_aug = augment_split(train_pairs)
val_pairs_aug   = augment_split(val_pairs)
test_pairs_aug  = augment_split(test_pairs)

Testing

In [ ]:
!pip install jellyfish

In [ ]:
from jellyfish import jaro_winkler_similarity

def build_features(pairs):
    features = pd.DataFrame()
    
    features["first_name_sim"] = pairs.apply(
        lambda r: jaro_winkler_similarity(
            str(r["first_name_1"]).lower(), 
            str(r["first_name_2"]).lower()
        ), axis=1
    )
    features["last_name_sim"] = pairs.apply(
        lambda r: jaro_winkler_similarity(
            str(r["last_name_1"]).lower(), 
            str(r["last_name_2"]).lower()
        ), axis=1
    )
    
    features["house_num_match"] = (pairs["house_num_1"] == pairs["house_num_2"]).astype(int)
    features["street_name_sim"] = pairs.apply(
        lambda r: jaro_winkler_similarity(
            str(r["street_name_1"]).lower(),
            str(r["street_name_2"]).lower()
        ), axis=1
    )
    features["street_type_match"] = (pairs["street_type_cd_1"] == pairs["street_type_cd_2"]).astype(int)
    
    return features

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train = build_features(train_pairs_aug)
y_train = train_pairs_aug["label"]

X_val = build_features(val_pairs_aug)
y_val = val_pairs_aug["label"]

# X_test = build_features(test_pairs_aug)
# y_test = test_pairs_aug["label"]

model = LogisticRegression()
model.fit(X_train, y_train)

print("Val results:")
print(classification_report(y_val, model.predict(X_val)))

# print("Test results:")
# print(classification_report(y_test, model.predict(X_test)))

In [ ]:
# from jellyfish import jaro_winkler_similarity

# def build_features(pairs):
#     features = pd.DataFrame()
    
#     # Name similarity - primary signal (almost never differs in real dups)
#     features["first_name_sim"] = pairs.apply(
#         lambda r: jaro_winkler_similarity(
#             str(r["first_name_1"]).lower(), 
#             str(r["first_name_2"]).lower()
#         ), axis=1
#     )
#     features["last_name_sim"] = pairs.apply(
#         lambda r: jaro_winkler_similarity(
#             str(r["last_name_1"]).lower(), 
#             str(r["last_name_2"]).lower()
#         ), axis=1
#     )
    
#     # Address similarity - noisy signal (91% differ in real dups)
#     features["house_num_match"] = (pairs["house_num_1"] == pairs["house_num_2"]).astype(int)
#     features["street_name_sim"] = pairs.apply(
#         lambda r: jaro_winkler_similarity(
#             str(r["street_name_1"]).lower(),
#             str(r["street_name_2"]).lower()
#         ), axis=1
#     )
#     features["street_type_match"] = (pairs["street_type_cd_1"] == pairs["street_type_cd_2"]).astype(int)
    
#     return features

In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import classification_report

# X_train = build_features(train_pairs_aug)
# y_train = train_pairs_aug["label"]

# X_val = build_features(val_pairs)
# y_val = val_pairs["label"]

# model = LogisticRegression()
# model.fit(X_train, y_train)

# y_pred = model.predict(X_val)
# print(classification_report(y_val, y_pred))

# # Check feature importance
# coefficients = pd.DataFrame({
#     "feature": X_train.columns,
#     "coefficient": model.coef_[0]
# }).sort_values("coefficient", ascending=False)
# print(coefficients)

In [ ]:
# print(val_pairs.shape)
# print(val_pairs["label"].value_counts(normalize=True))

In [ ]:
!git checkout Arpith

In [ ]:
def cosine_sim(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


def get_similarity(row1, row2):
    vec1 = train_embed.loc[row1].values
    vec2 = train_embed.loc[row2].values
    return cosine_sim(vec1, vec2)

In [ ]:
train_last_name = train_pairs_aug[
    (train_pairs_aug['label'] == 1) & 
    (train_pairs_aug['last_name_1'] != train_pairs_aug['last_name_2']) | (train_pairs_aug['label'] == 0)
].copy()

print(f"Number of duplicate records with different last names: {len(train_last_name)}")
print("\nSample of duplicate records with different last names:")
print(train_last_name.head())

val_last_name = val_pairs_aug[
    (val_pairs_aug['label'] == 1) & 
    (val_pairs_aug['last_name_1'] != val_pairs_aug['last_name_2']) | (val_pairs_aug['label'] == 0)
].copy()
    
print(f"Number of validation records with different last names: {len(val_last_name)}")
print("\nSample of validation records with different last names:")
print(val_last_name.head())

train_dup_pairs = train_last_name[train_last_name['label'] == 1][['id1', 'id2']].copy()
train_ndup_pairs = train_last_name[train_last_name['label'] == 0][['id1', 'id2']].copy()
train_last_name_pairs = train_last_name[['id1', 'id2', 'label']].copy()

train_last_name_pairs

In [ ]:
train_pairs_aug

In [ ]:
train_pairs_aug

In [ ]:
train_id1_data = train_pairs_aug[
    ['id_1', 'first_name_1', 'midl_name_1', 'last_name_1', 'age_1', 'sex_1', 'area_cd_1', 'zip_code_1', 'street_name_1', 'house_num_1', 'area_cd_1']
].rename(columns={
    'id_1': 'id',
    'first_name_1': 'first_name',
    'midl_name_1': 'middle_name',
    'last_name_1': 'last_name',
    'age_1': 'age',
    'sex_1': 'sex',
    'area_cd_1': 'district',
    'zip_code_1': 'zipcode',
    'street_name_1': 'street',
    'house_num_1': 'house_num',
    'area_cd_1':'area_cd'
    
})

train_id2_data = train_pairs_aug[
    ['id_2', 'first_name_2', 'midl_name_2', 'last_name_2', 'age_2', 'sex_2', 'area_cd_2', 'zip_code_2', 'street_name_2','house_num_2', 'area_cd_2']
].rename(columns={
    'id_2': 'id',
    'first_name_2': 'first_name',
    'midl_name_2': 'middle_name',
    'last_name_2': 'last_name',
    'age_2': 'age',
    'sex_2': 'sex',
    'area_cd_2': 'district',
    'zip_code_2': 'zipcode',
    'street_name_2': 'street',
    'house_num_2': 'house_num',
    'area_cd_2':'area_cd'
})

val_id1_data = val_pairs_aug[
    ['id_1', 'first_name_1', 'midl_name_1', 'last_name_1', 'age_1', 'sex_1', 'area_cd_1', 'zip_code_1', 'street_name_1','house_num_1','area_cd_1']
].rename(columns={
    'id_1': 'id',
    'first_name_1': 'first_name',
    'midl_name_1': 'middle_name',
    'last_name_1': 'last_name',
    'age_1': 'age',
    'sex_1': 'sex',
    'area_cd_1': 'district',
    'zip_code_1': 'zipcode',
    'street_name_1': 'street',
    'house_num_1': 'house_num',
    'area_cd_1':'area_cd'
})

val_id2_data = val_pairs_aug[
    ['id_2', 'first_name_2', 'midl_name_2', 'last_name_2', 'age_2', 'sex_2', 'area_cd_2', 'zip_code_2', 'street_name_2','house_num_2', 'area_cd_2']
].rename(columns={
    'id_2': 'id',
    'first_name_2': 'first_name',
    'midl_name_2': 'middle_name',
    'last_name_2': 'last_name',
    'age_2': 'age',
    'sex_2': 'sex',
    'area_cd_2': 'district',
    'zip_code_2': 'zipcode',
    'street_name_2': 'street',
    'house_num_2': 'house_num',
    'area_cd_2':'area_cd'
})

train_ids_data = pd.concat([train_id1_data, train_id2_data], ignore_index=True).drop_duplicates(subset=['id'], keep='first')
val_ids_data = pd.concat([val_id1_data, val_id2_data], ignore_index=True).drop_duplicates(subset=['id'], keep='first')

train_ids_data['middle_name'] = train_ids_data['middle_name'].fillna("")
val_ids_data['middle_name'] = val_ids_data['middle_name'].fillna("")

print(train_ids_data.head())
print(val_ids_data.head())
print(train_ids_data.columns.tolist())

In [ ]:
%cd C:\Users\arpit\spring-2026-deep-learning-fragmented-id-resolution\src

In [ ]:
from src.model import CharCNNEncoder
from src import model
from src import model_utils
from src.model_utils import get_embeddings, cosine_sim

In [ ]:
def evaluate_model(ids_data, pairs_data, vocab_size=38, embed_dim=32, num_filters=128, 
                   kernel_sizes=(3, 4, 5), dropout=0.3, output_dim=128, thresholds=None):
    """
    Evaluate embedding model with given parameters.
    
    Parameters:
    -----------
    ids_data : pd.DataFrame
        DataFrame with columns: id, first_name, middle_name, last_name, age, sex
    pairs_data : pd.DataFrame
        DataFrame with columns: id1, id2, label (1 for duplicate, 0 for non-duplicate)
    vocab_size : int
        Vocabulary size for character mapping
    embed_dim : int
        Character embedding dimension
    num_filters : int
        Number of CNN filters
    kernel_sizes : tuple
        Kernel sizes for CNN
    dropout : float
        Dropout rate
    output_dim : int
        Output embedding dimension
    thresholds : list
        Thresholds to evaluate (default: [0.92, 0.94, 0.96, 0.97, 0.975, 0.98, 0.99])
    
    Returns:
    --------
    dict with keys:
        - 'embeddings': Generated embeddings dataframe
        - 'similarities': Dataframe with pairs and their similarity scores
        - 'metrics_df': Metrics at each threshold
        - 'best_f1': Best F1 score
        - 'best_threshold': Threshold with best F1
    """
    if thresholds is None:
        thresholds = np.linspace(0.985,1.00,10)
    
    print("=" * 60)
    print(f"Model Configuration:")
    print(f"  vocab_size={vocab_size}, embed_dim={embed_dim}")
    print(f"  num_filters={num_filters}, kernel_sizes={kernel_sizes}")
    print(f"  dropout={dropout}, output_dim={output_dim}")
    print("=" * 60)
    
    # Create model
    model_eval = CharCNNEncoder(
        vocab_size=vocab_size,
        embed_dim=embed_dim,
        num_filters=num_filters,
        kernel_sizes=kernel_sizes,
        dropout=dropout,
        output_dim=output_dim
    )
    
    # Generate embeddings
    print("\n1. Generating embeddings...")
    embeddings = get_embeddings(ids_data, model_eval)
    embeddings_indexed = embeddings.set_index('id')
    print(f"   ✓ Generated embeddings shape: {embeddings.shape}")
    print(f"   ✓ NaN values: {embeddings.isnull().sum().sum()}")
    
    # Compute similarities
    print("\n2. Computing similarities...")
    similarities = []
    for idx, row in pairs_data.iterrows():
        id1, id2 = row['id1'], row['id2']
        try:
            vec1 = embeddings_indexed.loc[id1].values
            vec2 = embeddings_indexed.loc[id2].values
            sim = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
            similarities.append(sim)
        except KeyError:
            similarities.append(np.nan)
    
    pairs_with_sim = pairs_data.copy()
    pairs_with_sim['similarity'] = similarities
    print(f"   ✓ Computed {len(pairs_with_sim)} pair similarities")
    print(f"   ✓ Missing values: {pairs_with_sim['similarity'].isnull().sum()}")
    
    # Separate by label
    dup_sims = pairs_with_sim[pairs_with_sim['label'] == 1]['similarity'].dropna()
    ndup_sims = pairs_with_sim[pairs_with_sim['label'] == 0]['similarity'].dropna()
    
    print(f"\n3. Similarity Statistics:")
    print(f"   Duplicates (n={len(dup_sims)}): mean={dup_sims.mean():.4f}, std={dup_sims.std():.4f}")
    print(f"   Non-dups (n={len(ndup_sims)}): mean={ndup_sims.mean():.4f}, std={ndup_sims.std():.4f}")
    
    # Evaluate thresholds
    print(f"\n4. Evaluating thresholds...")
    metrics_results = []
    for thresh in thresholds:
        predictions = (pairs_with_sim['similarity'] > thresh).astype(int)
        actuals = pairs_with_sim['label'].astype(int)
        
        tn, fp, fn, tp = confusion_matrix(actuals, predictions).ravel()
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        
        metrics_results.append({
            'Threshold': thresh,
            'Precision': precision,
            'Recall': recall,
            'F1': f1,
            'Accuracy': accuracy,
            'TP': tp,
            'FP': fp,
            'FN': fn,
            'TN': tn
        })
    
    metrics_df = pd.DataFrame(metrics_results)
    best_f1_idx = metrics_df['F1'].idxmax()
    best_f1 = metrics_df.loc[best_f1_idx, 'F1']
    best_threshold = metrics_df.loc[best_f1_idx, 'Threshold']
    
    print(f"\n   Best F1: {best_f1:.4f} at threshold {best_threshold:.3f}")
    
    # Create visualizations
    print(f"\n5. Creating visualizations...")
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    
    # Histogram
    ax1.hist(dup_sims, bins=50, alpha=0.6, label='Duplicates', color='green', edgecolor='black')
    ax1.hist(ndup_sims, bins=50, alpha=0.6, label='Non-Duplicates', color='red', edgecolor='black')
    ax1.set_xlabel('Cosine Similarity', fontsize=11)
    ax1.set_ylabel('Frequency', fontsize=11)
    ax1.set_title('Distribution of Embedding Similarities', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # Precision-Recall Curve
    precisions_pr, recalls_pr, _ = precision_recall_curve(
        pairs_with_sim['label'].astype(int),
        pairs_with_sim['similarity']
    )
    ax2.plot(recalls_pr, precisions_pr, linewidth=2, color='blue')
    ax2.scatter(metrics_df['Recall'], metrics_df['Precision'], c=metrics_df['Threshold'], 
                cmap='viridis', s=100, zorder=5)
    ax2.set_xlabel('Recall', fontsize=11)
    ax2.set_ylabel('Precision', fontsize=11)
    ax2.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # F1 vs Threshold
    ax3.plot(metrics_df['Threshold'], metrics_df['F1'], 'o-', linewidth=2, 
             markersize=8, label='F1', color='blue')
    ax3.plot(metrics_df['Threshold'], metrics_df['Precision'], 's-', linewidth=2, 
             markersize=6, label='Precision', color='orange')
    ax3.plot(metrics_df['Threshold'], metrics_df['Recall'], '^-', linewidth=2, 
             markersize=6, label='Recall', color='green')
    ax3.axvline(best_threshold, color='red', linestyle='--', alpha=0.7, label='Best F1')
    ax3.set_xlabel('Threshold', fontsize=11)
    ax3.set_ylabel('Score', fontsize=11)
    ax3.set_title('Metrics vs Similarity Threshold', fontsize=12, fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Confusion Matrix at best threshold
    best_preds = (pairs_with_sim['similarity'] > best_threshold).astype(int)
    best_cm = confusion_matrix(pairs_with_sim['label'].astype(int), best_preds)
    sns.heatmap(best_cm, annot=True, fmt='d', cmap='Blues', ax=ax4, cbar=False,
                xticklabels=['Non-Dup', 'Dup'], yticklabels=['Non-Dup', 'Dup'])
    ax4.set_ylabel('Actual', fontsize=11)
    ax4.set_xlabel('Predicted', fontsize=11)
    ax4.set_title(f'Confusion Matrix (threshold={best_threshold:.3f}, F1={best_f1:.4f})', 
                  fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n" + "=" * 60)
    print("THRESHOLD EVALUATION RESULTS:")
    print("=" * 60)
    print(metrics_df.to_string(index=False))
    print("=" * 60)
    
    return {
        'embeddings': embeddings,
        'similarities': pairs_with_sim,
        'metrics_df': metrics_df,
        'best_f1': best_f1,
        'best_threshold': best_threshold,
        'dup_sims': dup_sims,
        'ndup_sims': ndup_sims
    }

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_curve

import matplotlib.pyplot as plt

# Define the dimensions to test
embed_dims = [8, 16, 32, 64]
output_dims = [16, 32, 64, 128]

# Collect results in a list
results_list = []

for embed in embed_dims:
    for output in output_dims:
        print(f"Evaluating embed_dim={embed}, output_dim={output}")
        result = evaluate_model(
            ids_data=train_ids_data,
            pairs_data=train_last_name_pairs,
            vocab_size=38,
            embed_dim=embed,
            num_filters=128,
            kernel_sizes=(3, 4, 5),
            dropout=0.3,
            output_dim=output
        )
        results_list.append({
            'embed_dim': embed,
            'output_dim': output,
            'best_f1': result['best_f1']
        })

# Create a DataFrame from results
results_df = pd.DataFrame(results_list)

# Pivot to create a grid (embed_dim as rows, output_dim as columns)
f1_grid = results_df.pivot(index='embed_dim', columns='output_dim', values='best_f1')

# Display the grid as a table
print("F1 Scores Grid (embed_dim vs output_dim):")
print(f1_grid)

# Visualize as a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(f1_grid, annot=True, fmt=".4f", cmap="viridis", cbar_kws={'label': 'F1 Score'})
plt.title('F1 Scores Heatmap: embed_dim vs output_dim')
plt.xlabel('output_dim')
plt.ylabel('embed_dim')
plt.show()

In [ ]:
# move to repo root (you are currently in /src)
%cd ..

# confirm branch, switch if needed
!git branch --show-current
!git checkout Arpith

# stage, commit, and push to Arpith
!git add -A
!git commit -m "Updated F1 score evaluations"
!git push origin Arpith

## Trying on the whole data

We apply the exact same thing on the entire data and see how F1 score changes

In [ ]:
%cd "C:\Users\arpit\spring-2026-deep-learning-fragmented-id-resolution"

In [ ]:
train_data = pd.read_csv("data/train_pairs_aug.csv", delimiter=',')
val_data = pd.read_csv("data/val_pairs_aug.csv")

train_id1_data = train_data[
    ['id_1', 'first_name_1', 'midl_name_1', 'last_name_1', 'age_1', 'sex_1', 'area_cd_1', 'zip_code_1', 'street_name_1', 'house_num_1', 'area_cd_1']
].rename(columns={
    'id_1': 'id',
    'first_name_1': 'first_name',
    'midl_name_1': 'middle_name',
    'last_name_1': 'last_name',
    'age_1': 'age',
    'sex_1': 'sex',
    'area_cd_1': 'district',
    'zip_code_1': 'zipcode',
    'street_name_1': 'street',
    'house_num_1': 'house_num',
    'area_cd_1':'area_cd'
    
})

train_id2_data = train_data[
    ['id_2', 'first_name_2', 'midl_name_2', 'last_name_2', 'age_2', 'sex_2', 'area_cd_2', 'zip_code_2', 'street_name_2','house_num_2', 'area_cd_2']
].rename(columns={
    'id_2': 'id',
    'first_name_2': 'first_name',
    'midl_name_2': 'middle_name',
    'last_name_2': 'last_name',
    'age_2': 'age',
    'sex_2': 'sex',
    'area_cd_2': 'district',
    'zip_code_2': 'zipcode',
    'street_name_2': 'street',
    'house_num_2': 'house_num',
    'area_cd_2':'area_cd'
})

val_id1_data = val_data[
    ['id_1', 'first_name_1', 'midl_name_1', 'last_name_1', 'age_1', 'sex_1', 'area_cd_1', 'zip_code_1', 'street_name_1','house_num_1','area_cd_1']
].rename(columns={
    'id_1': 'id',
    'first_name_1': 'first_name',
    'midl_name_1': 'middle_name',
    'last_name_1': 'last_name',
    'age_1': 'age',
    'sex_1': 'sex',
    'area_cd_1': 'district',
    'zip_code_1': 'zipcode',
    'street_name_1': 'street',
    'house_num_1': 'house_num',
    'area_cd_1':'area_cd'
})

val_id2_data = val_data[
    ['id_2', 'first_name_2', 'midl_name_2', 'last_name_2', 'age_2', 'sex_2', 'area_cd_2', 'zip_code_2', 'street_name_2','house_num_2', 'area_cd_2']
].rename(columns={
    'id_2': 'id',
    'first_name_2': 'first_name',
    'midl_name_2': 'middle_name',
    'last_name_2': 'last_name',
    'age_2': 'age',
    'sex_2': 'sex',
    'area_cd_2': 'district',
    'zip_code_2': 'zipcode',
    'street_name_2': 'street',
    'house_num_2': 'house_num',
    'area_cd_2':'area_cd'
})

train_ids_data = pd.concat([train_id1_data, train_id2_data], ignore_index=True).drop_duplicates(subset=['id'], keep='first')
val_ids_data = pd.concat([val_id1_data, val_id2_data], ignore_index=True).drop_duplicates(subset=['id'], keep='first')

train_ids_data['middle_name'] = train_ids_data['middle_name'].fillna("")
val_ids_data['middle_name'] = val_ids_data['middle_name'].fillna("")

print(train_ids_data.head())
print(val_ids_data.head())
print(train_ids_data.columns.tolist())

In [ ]:
# Create full pairs dataframe with labels
train_pairs_full = train_data[['id_1', 'id_2', 'label']].rename(columns={'id_1': 'id1', 'id_2': 'id2'})
val_pairs_full = val_data[['id_1', 'id_2', 'label']].rename(columns={'id_1': 'id1', 'id_2': 'id2'})

print(f"Full training pairs: {len(train_pairs_full)}")
print(f"  Duplicates: {(train_pairs_full['label'] == 1).sum()}")
print(f"  Non-duplicates: {(train_pairs_full['label'] == 0).sum()}")
print(f"\nFull validation pairs: {len(val_pairs_full)}")
print(f"  Duplicates: {(val_pairs_full['label'] == 1).sum()}")
print(f"  Non-duplicates: {(val_pairs_full['label'] == 0).sum()}")


In [ ]:
# Grid search over embedding dimensions on FULL augmented data
print("\n" + "="*70)
print("GRID SEARCH: Embedding vs Output Dimensions (FULL AUGMENTED DATA)")
print("="*70)

embed_dims = [8, 16, 32, 64]
output_dims = [16, 32, 64, 128]

results_list_full = []

for embed in embed_dims:
    for output in output_dims:
        print(f"\n{'='*70}")
        print(f"Testing embed_dim={embed}, output_dim={output}")
        print(f"{'='*70}")
        
        result = evaluate_model(
            ids_data=train_ids_data,
            pairs_data=train_pairs_full,
            vocab_size=38,
            embed_dim=embed,
            num_filters=128,
            kernel_sizes=(3, 4, 5),
            dropout=0.3,
            output_dim=output
        )
        
        results_list_full.append({
            'embed_dim': embed,
            'output_dim': output,
            'best_f1': result['best_f1'],
            'best_threshold': result['best_threshold']
        })
        
        print(f"\n✓ Best F1: {result['best_f1']:.4f} at threshold {result['best_threshold']:.4f}")

# Create results dataframe
results_df_full = pd.DataFrame(results_list_full)

# Pivot to create a grid
f1_grid_full = results_df_full.pivot(index='embed_dim', columns='output_dim', values='best_f1')

print("\n" + "="*70)
print("F1 SCORES GRID - FULL AUGMENTED DATA (embed_dim vs output_dim):")
print("="*70)
print(f1_grid_full)

# Visualize as heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(f1_grid_full, annot=True, fmt=".4f", cmap="viridis", 
            cbar_kws={'label': 'F1 Score'}, linewidths=0.5)
plt.title('F1 Scores Heatmap: Full Augmented Data\n(embed_dim vs output_dim)', 
          fontsize=14, fontweight='bold')
plt.xlabel('output_dim', fontsize=12)
plt.ylabel('embed_dim', fontsize=12)
plt.tight_layout()
plt.savefig('f1_grid_full_augmented.png', dpi=150, bbox_inches='tight')
plt.show()

# Show best configuration
best_idx = results_df_full['best_f1'].idxmax()
best_config = results_df_full.loc[best_idx]
print(f"\n{'='*70}")
print("BEST CONFIGURATION (FULL DATA):")
print(f"{'='*70}")
print(f"  embed_dim: {int(best_config['embed_dim'])}")
print(f"  output_dim: {int(best_config['output_dim'])}")
print(f"  Best F1: {best_config['best_f1']:.4f}")
print(f"  Best Threshold: {best_config['best_threshold']:.4f}")
print(f"{'='*70}")
